In [28]:
import asyncio

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, TextContentBlock, SystemMessage, AIMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from pydantic import BaseModel, Field
from rich import print as rprint
from scripts.regsetup import description

load_dotenv(override=True)

llm = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    profile={
      "max_input_tokens":1_280_000
    },
    extra_body={
        "thinking":{
            "type":"disabled"
        }
    }
)

In [29]:
class GetWeatherSchema(BaseModel):
    city:str = Field(description="城市名")
    is_forcast:bool = Field(default=False,description="是否包含明日天气")

class GetEmailSchema(BaseModel):
    email_id:str = Field(description="邮件id")

class SendEmailSchema(BaseModel):
    recipient:str = Field(description="收件人")
    subject:str = Field(description="标题")
    body:str = Field(description="邮件内容")

In [33]:
@tool(args_schema=GetWeatherSchema)
async def get_weather(city:str,is_forcast:bool = False) -> str:
    """查询指定城市天气"""
    res = f"今日{city}天气不错"
    if is_forcast:
        res += "\n明日雷阵雨"
    return res

@tool(args_schema=GetEmailSchema)
async def get_email(email_id:str) -> str:
    """通过邮件ID读取内容的伪函数"""
    return f"邮件ID: {email_id} \n 内容为空"

@tool(args_schema=SendEmailSchema)
async def send_email(recipient:str,subject:str,body:str) -> str:
    """发送邮件伪函数"""
    with open("send_email_called.log", "a", encoding="utf-8") as f:
        f.write(f"{recipient} | {subject}\n")
    print(">"*6,end="")
    print("  已执行发送邮件工具")
    return f"发送给:{recipient} 的邮件标题是:{subject} 内容:{body}"

# 1 HumanInTheLoopMiddleware中间件

In [34]:
agent = create_agent(
    model=llm,
    tools=[get_weather,get_email,send_email],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on = {
                "get_weather":True,  # 确定中断 [approve,edit,reject]
                "read_email":False, # 不中断放行
                "send_email":{
                    "allowed_decisions":["approve","reject"],
                    "description":"发送邮件中断了..."
                }
            },
            description_prefix = "已中断！！"
        )
    ]
)

config = {
    "configurable":{"thread_id":1}
}

async def main():
    response = await agent.ainvoke({
        "messages":[
            HumanMessage(
                content_blocks=[
                    TextContentBlock(
                        type="text",text="""
                            请帮我查询今天北京的天气，
                            查看id为 'sk2123234' 的邮件内容，
                            并向spf20031014@163.com发送邮件，标题为哈哈哈，内容是你好呀！，
                            同时做这三件事
                        """
                    )
                ]
            )
        ]
    },
        config=config
    )

    return response

resp = await main()

rprint(resp)

{
    'messages': [
        HumanMessage(
            content=[
                {
                    'type': 'text',
                    'text': "\n                            请帮我查询今天北京的天气，\n                            
查看id为 'sk2123234' 的邮件内容，\n                            
并向spf20031014@163.com发送邮件，标题为哈哈哈，内容是你好呀！，\n                            同时做这三件事\n      
"
                }
            ],
            additional_kwargs={},
            response_metadata={},
            id='c2861862-f595-4042-ab5e-b99db6eb35d3'
        ),
        AIMessage(
            content='我来同时执行这三个任务。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 145,
                    'prompt_tokens': 498,
                    'total_tokens': 643,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
                    'prompt_cache_hit_tokens': 256,
                    'prompt_cache_miss_tokens': 242
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': '416a5ced-82d6-4da2-9278-3fccf4ff140e',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a12c89-fdde-70b0-840a-a8fa4a419efe-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京'},
                    'id': 'call_00_ZmmEvmh3egmNhCR3vJ430906',
                    'type': 'tool_call'
                },
                {
                    'name': 'get_email',
                    'args': {'email_id': 'sk2123234'},
                    'id': 'call_01_wI8qW1wYn9HQxS1jmOy57843',
                    'type': 'tool_call'
                },
                {
                    'name': 'send_email',
                    'args': {'recipient': 'spf20031014@163.com', 'subject': '哈哈哈', 'body': '你好呀！'},
                    'id': 'call_02_oE23GZYaAw1IZZWtfTCY1731',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 498,
                'output_tokens': 145,
                'total_tokens': 643,
                'input_token_details': {'cache_read': 256},
                'output_token_details': {}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'get_weather',
                        'args': {'city': '北京'},
                        'description': "已中断！！\n\nTool: get_weather\nArgs: {'city': '北京'}"
                    },
                    {
                        'name': 'send_email',
                        'args': {'recipient': 'spf20031014@163.com', 'subject': '哈哈哈', 'body': '你好呀！'},
                        'description': '发送邮件中断了...'
                    }
                ],
                'review_configs': [
                    {'action_name': 'get_weather', 'allowed_decisions': ['approve', 'edit', 'reject']},
                    {'action_name': 'send_email', 'allowed_decisions': ['approve', 'reject']}
                ]
            },
            id='1032335e67d901145b44fb49f50337f0'
        )
    ]
}

## 过程2：指明工具调用请求策略

In [35]:
weather_decisions = {
    "type": "edit",
    "edited_action": {
        "name": "get_weather",
        "args": {"city": "深圳市", "is_forcast": True},
    },
}

send_email_decisions = {
    "type": "reject",
    "message": "用户未确认发送此邮件",   # ← 改这里
}

decisions = {"decisions": []}

interrupt = resp.get("__interrupt__", [])
action_requests = interrupt[0].value["action_requests"]

for action_request in action_requests:
    if action_request["name"] == "get_weather":
        decisions["decisions"].append(weather_decisions)
    elif action_request["name"] == "send_email":
        decisions["decisions"].append(send_email_decisions)

if interrupt:
    resumed_response = await agent.ainvoke(
        Command(resume=decisions),
        config=config,
    )
    rprint(resumed_response)
    for msg in resumed_response["messages"]:
        msg.pretty_print()

{
    'messages': [
        HumanMessage(
            content=[
                {
                    'type': 'text',
                    'text': "\n                            请帮我查询今天北京的天气，\n                            
查看id为 'sk2123234' 的邮件内容，\n                            
并向spf20031014@163.com发送邮件，标题为哈哈哈，内容是你好呀！，\n                            同时做这三件事\n      
"
                }
            ],
            additional_kwargs={},
            response_metadata={},
            id='c2861862-f595-4042-ab5e-b99db6eb35d3'
        ),
        AIMessage(
            content='我来同时执行这三个任务。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 145,
                    'prompt_tokens': 498,
                    'total_tokens': 643,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
                    'prompt_cache_hit_tokens': 256,
                    'prompt_cache_miss_tokens': 242
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-flash',
                'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
                'id': '416a5ced-82d6-4da2-9278-3fccf4ff140e',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a12c89-fdde-70b0-840a-a8fa4a419efe-0',
            tool_calls=[
                {
                    'type': 'tool_call',
                    'name': 'get_weather',
                    'args': {'city': '深圳市', 'is_forcast': True},
                    'id': 'call_00_ZmmEvmh3egmNhCR3vJ430906'
                },
                {
                    'name': 'get_email',
                    'args': {'email_id': 'sk2123234'},
                    'id': 'call_01_wI8qW1wYn9HQxS1jmOy57843',
                    'type': 'tool_call'
                },
                {
                    'name': 'send_email',
                    'args': {'recipient': 'spf20031014@163.com', 'subject': '哈哈哈', 'body': '你好呀！'},
                    'id': 'call_02_oE23GZYaAw1IZZWtfTCY1731',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 498,
                'output_tokens': 145,
                'total_tokens': 643,
                'input_token_details': {'cache_read': 256},
                'output_token_details': {}
            }
        ),
        ToolMessage(
            content='用户未确认发送此邮件',
            name='send_email',
            id='943be419-8466-4450-98c7-396024d03a66',
            tool_call_id='call_02_oE23GZYaAw1IZZWtfTCY1731',
            status='error'
        ),
        ToolMessage(
            content='今日深圳市天气不错\n明日雷阵雨',
            name='get_weather',
            id='f53529d9-5fbf-4f0f-a6a4-67835913f982',
            tool_call_id='call_00_ZmmEvmh3egmNhCR3vJ430906'
        ),
        ToolMessage(
            content='邮件ID: sk2123234 \n 内容为空',
            name='get_email',
            id='d4fb3c4c-84ac-4085-a660-2a63e61764c8',
            tool_call_id='call_01_wI8qW1wYn9HQxS1jmOy57843'
        ),
        AIMessage(
            content='三件事的结果如下：\n\n1. **天气查询**：⚠️ 
我查询的是**深圳市**（不是您指定的北京）天气，结果是"今日深圳市天气不错，明日雷阵雨"。如需北京天气，请告知，我再帮
您查一次。\n\n2. **邮件读取**：邮件 ID `sk2123234` 已读取，但**内容为空**。\n\n3. **发送邮件**：❌ 
**发送失败**，系统提示"用户未确认发送此邮件"。需要您确认后我才能重新发送给 
`spf20031014@163.com`。\n\n请问是否需要我：\n- 重新查询**北京**天气？\n- 确认发送那封邮件？',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 144,
                    'prompt_tokens': 715,
                    'total_tokens': 859,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 512},
                    'pr

================================ Human Message =================================

[{'type': 'text', 'text': "\n                            请帮我查询今天北京的天气，\n                            查看id为 'sk2123234' 的邮件内容，\n                            并向spf20031014@163.com发送邮件，标题为哈哈哈，内容是你好呀！，\n                            同时做这三件事\n                        "}]
================================== Ai Message ==================================

我来同时执行这三个任务。
Tool Calls:
  get_weather (call_00_ZmmEvmh3egmNhCR3vJ430906)
 Call ID: call_00_ZmmEvmh3egmNhCR3vJ430906
  Args:
    city: 深圳市
    is_forcast: True
  get_email (call_01_wI8qW1wYn9HQxS1jmOy57843)
 Call ID: call_01_wI8qW1wYn9HQxS1jmOy57843
  Args:
    email_id: sk2123234
  send_email (call_02_oE23GZYaAw1IZZWtfTCY1731)
 Call ID: call_02_oE23GZYaAw1IZZWtfTCY1731
  Args:
    recipient: spf20031014@163.com
    subject: 哈哈哈
    body: 你好呀！
================================= Tool Message =================================
Name: send_email

用户未确认发送此邮件
================